# Evidencia 1 — Base de Datos Big Data  
## Muestreo y particionamiento con PySpark

**Dataset:** *eCommerce behavior data from multi category store* — Kaggle  
**Equipo:** Equipo #27   
**Integrantes:**  
- Jaime A. Mendivil Altamirano — A01253316  
- Antonio Ramón Valerio Tejada — A01797448  
- Carlos Enrique García Díaz — A01570123  
- Rodolfo González Villaseñor — A01652430  

**Objetivo:** construir una muestra representativa de la población objetivo mediante caracterización, particionamiento y muestreo con PySpark.


## 1. Contexto de la actividad

La población objetivo corresponde a eventos de interacción de usuarios dentro de una tienda electrónica multicategoría. Cada fila del dataset representa un evento generado por un usuario durante una sesión de navegación o compra.  

La actividad se estructura en cuatro partes:

1. **Caracterización de la población:** identificación y descripción de variables relevantes.
2. **Particionamiento:** construcción de reglas de partición a partir de variables de caracterización.
3. **Implementación en PySpark:** extracción de submuestras por regla de particionamiento.
4. **Técnica de muestreo:** definición y justificación de la técnica para recuperar instancias representativas de cada partición.

> Nota: este notebook está preparado para trabajar con archivos CSV del dataset de Kaggle. Por el tamaño del dataset, se recomienda usar PySpark y leer solo las columnas necesarias cuando sea posible.

## 2. Instalación y configuración

Esta versión fue ajustada con base en el notebook anterior que ya funcionó. Se conserva el uso de `findspark`, `kagglehub` y la lectura directa del archivo `2019-Oct.csv`.


In [1]:
# Si estás en Colab o en un entorno nuevo, descomenta las siguientes líneas:
# !pip install -q pyspark findspark kagglehub

import os
import math
from pathlib import Path

import findspark
findspark.init()
findspark.find()

from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Muestreo_BigData_Ecommerce")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

# Configuración para visualizar mejor los DataFrames en notebooks.
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark


## 3. Descarga del dataset con KaggleHub

Se utiliza el mismo método del notebook anterior: `kagglehub.dataset_download`. Esto evita configurar manualmente `kaggle.json` y permite descargar directamente el dataset de Kaggle.


In [3]:
# Importar la biblioteca de KaggleHub
import kagglehub

# Descargar el dataset de Kaggle
path = kagglehub.dataset_download("mkechinov/ecommerce-behavior-data-from-multi-category-store")

# Construir la ruta al archivo CSV de octubre de 2019
file_path = f"{path}/2019-Oct.csv"
print("Archivo utilizado:", file_path)

# Leer el archivo CSV con las opciones que funcionaron en el notebook anterior
df = spark.read.csv(
    file_path,
    header=True,
    sep=",",
    encoding="iso-8859-1",
    inferSchema=True
)

# Reducir el tamaño desde el inicio para trabajar de forma estable en Windows local.
# Para una computadora con 8 GB RAM puedes bajar a 0.005; con 16 GB normalmente 0.01 funciona.
SAMPLE_INPUT_FRACTION = 0.01
SEED = 42

df = df.sample(False, SAMPLE_INPUT_FRACTION, seed=SEED)

print(f"Se trabajará con una muestra inicial aproximada del {SAMPLE_INPUT_FRACTION*100:.1f}% del CSV original.")
df.printSchema()
df.show(5, truncate=False)


Archivo utilizado: C:\Users\YOLOA\.cache\kagglehub\datasets\mkechinov\ecommerce-behavior-data-from-multi-category-store\versions\8/2019-Oct.csv
Se trabajará con una muestra inicial aproximada del 1.0% del CSV original.
root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)

+-------------------+----------+----------+-------------------+----------------------+-------+------+---------+------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code         |brand  |price |user_id  |user_session                        |
+-------------------+----------+----------+-------------------+----------------------+-------+------+--

## 4. Revisión y ajuste de tipos de datos

Aunque algunos identificadores se leen como variables numéricas, se tratan como identificadores categóricos. Por ello se ajustan a tipos con mayor capacidad (`long`). Para el precio se utiliza `DecimalType(10,2)` porque representa cantidades monetarias con dos decimales.


In [4]:
df.printSchema()

# Se corrigen identificadores a long porque pueden existir muchos productos, categorías y usuarios.
df = df.withColumn("product_id", F.col("product_id").cast("long"))
df = df.withColumn("category_id", F.col("category_id").cast("long"))
df = df.withColumn("user_id", F.col("user_id").cast("long"))

# Se corrige el precio a decimal para representar valores monetarios.
df = df.withColumn("price", F.col("price").cast(DecimalType(10, 2)))

print("Schema después de ajustar tipos:")
df.printSchema()


root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)

Schema después de ajustar tipos:
root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_session: string (nullable = true)



In [5]:
print("Vista rápida de registros cargados:")
df.show(5, truncate=False)

# Conteo sobre la muestra inicial, no sobre el CSV completo.
print(f"Registros disponibles en la muestra inicial: {df.count():,}")


Vista rápida de registros cargados:
+-------------------+----------+----------+-------------------+----------------------+-------+------+---------+------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code         |brand  |price |user_id  |user_session                        |
+-------------------+----------+----------+-------------------+----------------------+-------+------+---------+------------------------------------+
|2019-09-30 17:03:38|view      |5300650   |2053013563173241677|NULL                  |vitek  |12.84 |552584835|b56e1552-5a77-44bc-af49-d1b1ecd00b4e|
|2019-09-30 17:04:45|view      |1004870   |2053013555631882655|electronics.smartphone|samsung|286.86|513881159|387ba75c-47bb-480a-9f86-b7c5cc57cf6b|
|2019-09-30 17:06:19|view      |1004246   |2053013555631882655|electronics.smartphone|apple  |736.18|533940457|309e5dba-628b-491a-98ec-4c2906db831a|
|2019-09-30 17:11:38|view      |9300040   |2053013554524586339|NULL   

## 5. Limpieza mínima y variables derivadas

Se aplican transformaciones mínimas para evitar trabajar con registros incompletos en variables críticas. Además, se crean variables de caracterización que ayudarán a construir particiones:

- `event_date`: fecha del evento.
- `hour`: hora del día.
- `day_of_week`: día de la semana.
- `category_level_1`: categoría principal extraída de `category_code`.
- `price_segment`: segmento de precio.

> Nota técnica: se eliminó el cálculo de `session_event_count` porque requiere un `groupBy` y `join` sobre muchas sesiones, lo que puede provocar errores de memoria en Spark local sobre Windows.


In [6]:
# Revisión básica de nulos antes de limpiar
print("Valores nulos antes de la limpieza:")
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

# Limpieza mínima y creación de variables derivadas
# Se evitan operaciones pesadas como join por sesión para reducir consumo de memoria.
df_clean = (
    df
    .dropna(subset=["event_time", "event_type", "product_id", "price", "user_id", "user_session"])
    .filter(F.col("price") >= 0)
    .withColumn("event_date", F.to_date("event_time"))
    .withColumn("hour", F.hour("event_time"))
    .withColumn("day_of_week", F.date_format("event_time", "E"))
    .withColumn(
        "category_level_1",
        F.when(F.col("category_code").isNull(), "unknown")
         .otherwise(F.split(F.col("category_code"), "\\.").getItem(0))
    )
)

# Segmentos de precio usando cuantiles aproximados.
q25, q50, q75 = df_clean.approxQuantile("price", [0.25, 0.50, 0.75], 0.05)

df_clean = (
    df_clean
    .withColumn(
        "price_segment",
        F.when(F.col("price") <= q25, "bajo")
         .when(F.col("price") <= q50, "medio_bajo")
         .when(F.col("price") <= q75, "medio_alto")
         .otherwise("alto")
    )
)

print("Muestra de datos limpios:")
df_clean.show(5, truncate=False)
print(f"Registros limpios en la muestra de trabajo: {df_clean.count():,}")


Valores nulos antes de la limpieza:
+----------+----------+----------+-----------+-------------+-----+-----+-------+------------+
|event_time|event_type|product_id|category_id|category_code|brand|price|user_id|user_session|
+----------+----------+----------+-----------+-------------+-----+-----+-------+------------+
|         0|         0|         0|          0|       135517|61139|    0|      0|           0|
+----------+----------+----------+-----------+-------------+-----+-----+-------+------------+

Muestra de datos limpios:
+-------------------+----------+----------+-------------------+----------------------+-------+------+---------+------------------------------------+----------+----+-----------+----------------+-------------+
|event_time         |event_type|product_id|category_id        |category_code         |brand  |price |user_id  |user_session                        |event_date|hour|day_of_week|category_level_1|price_segment|
+-------------------+----------+----------+--------

## 6. Caracterización de la población

La siguiente tabla documenta variables de caracterización relevantes para entender el comportamiento de la población objetivo. Esta sección responde al criterio de la rúbrica relacionado con describir cada variable, su dominio, estadísticos y comentarios.

In [7]:
caracterizacion = [
    ("event_type", "Categórica nominal", "view, cart, purchase", 
     "Frecuencia por tipo de interacción", 
     "Permite distinguir niveles de intención del usuario."),
    ("price", "Numérica continua", "Valores >= 0", 
     "media, mínimo, máximo y cuantiles", 
     "Permite segmentar productos por nivel de precio."),
    ("category_level_1", "Categórica nominal", "Categorías principales del catálogo", 
     "frecuencia por categoría", 
     "Permite comparar comportamiento entre familias de productos."),
    ("price_segment", "Categórica ordinal", "bajo, medio_bajo, medio_alto, alto", 
     "frecuencia por segmento", 
     "Construida a partir de cuantiles para estratificar por valor económico."),
    ("hour", "Numérica discreta", "0 a 23", 
     "frecuencia por hora", 
     "Permite observar temporalidad diaria del comportamiento."),
    ("day_of_week", "Categórica ordinal/temporal", "Mon, Tue, Wed...", 
     "frecuencia por día", 
     "Permite analizar variaciones por día de la semana."),
]

caracterizacion_df = spark.createDataFrame(
    caracterizacion,
    ["variable", "tipo", "dominio", "estadisticos_relevantes", "comentario"]
)

caracterizacion_df.show(truncate=False)


+----------------+---------------------------+-----------------------------------+----------------------------------+-----------------------------------------------------------------------+
|variable        |tipo                       |dominio                            |estadisticos_relevantes           |comentario                                                             |
+----------------+---------------------------+-----------------------------------+----------------------------------+-----------------------------------------------------------------------+
|event_type      |Categórica nominal         |view, cart, purchase               |Frecuencia por tipo de interacción|Permite distinguir niveles de intención del usuario.                   |
|price           |Numérica continua          |Valores >= 0                       |media, mínimo, máximo y cuantiles |Permite segmentar productos por nivel de precio.                       |
|category_level_1|Categórica nominal         |Cate

In [8]:
# Estadísticos numéricos ligeros. Se evita summary() para reducir carga en Spark local.
df_clean.select(
    F.mean("price").alias("precio_promedio"),
    F.min("price").alias("precio_minimo"),
    F.max("price").alias("precio_maximo"),
    F.mean("hour").alias("hora_promedio"),
    F.min("hour").alias("hora_minima"),
    F.max("hour").alias("hora_maxima")
).show()

# Frecuencias de variables categóricas clave
for col in ["event_type", "price_segment", "category_level_1", "day_of_week"]:
    print(f"\nFrecuencia de {col}")
    df_clean.groupBy(col).count().orderBy(F.desc("count")).show(10, truncate=False)


+---------------+-------------+-------------+-----------------+-----------+-----------+
|precio_promedio|precio_minimo|precio_maximo|    hora_promedio|hora_minima|hora_maxima|
+---------------+-------------+-------------+-----------------+-----------+-----------+
|     291.563160|         0.00|      2574.07|9.759624472752408|          0|         23|
+---------------+-------------+-------------+-----------------+-----------+-----------+


Frecuencia de event_type
+----------+------+
|event_type|count |
+----------+------+
|view      |408827|
|cart      |9110  |
|purchase  |7385  |
+----------+------+


Frecuencia de price_segment
+-------------+------+
|price_segment|count |
+-------------+------+
|alto         |120992|
|bajo         |107066|
|medio_alto   |101094|
|medio_bajo   |96170 |
+-------------+------+


Frecuencia de category_level_1
+----------------+------+
|category_level_1|count |
+----------------+------+
|electronics     |162028|
|unknown         |135517|
|appliances     

## 7. Definición de variables para particionamiento

Para construir particiones útiles y representativas, se seleccionan tres variables:

1. **`event_type`**: representa la intención del usuario.
2. **`price_segment`**: representa el nivel económico del producto.
3. **`category_level_1`**: representa la familia principal del producto.

Estas variables son adecuadas porque combinan comportamiento, valor económico y tipo de producto. Además, permiten construir subconjuntos interpretables, por ejemplo: compras de precio alto en electrónica, vistas de precio bajo en ropa, etc.

In [9]:
# Para evitar demasiadas particiones, se conservan las categorías principales más frecuentes.
TOP_K_CATEGORIES = 8

top_categories = (
    df_clean.groupBy("category_level_1")
    .count()
    .orderBy(F.desc("count"))
    .limit(TOP_K_CATEGORIES)
    .select("category_level_1")
)

top_categories_list = [row["category_level_1"] for row in top_categories.collect()]
top_categories_list

['electronics',
 'unknown',
 'appliances',
 'computers',
 'apparel',
 'furniture',
 'auto',
 'construction']

In [10]:
df_part = (
    df_clean
    .withColumn("brand", F.coalesce(F.col("brand"), F.lit("unknown")))
    .withColumn("category_partition",
                F.when(F.col("category_level_1").isin(top_categories_list), F.col("category_level_1"))
                 .otherwise(F.lit("otras")))
    .withColumn(
        "partition_rule",
        F.concat_ws(
            " | ",
            F.concat(F.lit("event_type="), F.col("event_type")),
            F.concat(F.lit("price_segment="), F.col("price_segment")),
            F.concat(F.lit("category="), F.col("category_partition"))
        )
    )
)

df_part.select("event_type", "price_segment", "category_partition", "partition_rule").show(10, truncate=False)

+----------+-------------+------------------+-----------------------------------------------------------------+
|event_type|price_segment|category_partition|partition_rule                                                   |
+----------+-------------+------------------+-----------------------------------------------------------------+
|view      |bajo         |unknown           |event_type=view | price_segment=bajo | category=unknown          |
|view      |medio_alto   |electronics       |event_type=view | price_segment=medio_alto | category=electronics|
|view      |alto         |electronics       |event_type=view | price_segment=alto | category=electronics      |
|view      |alto         |unknown           |event_type=view | price_segment=alto | category=unknown          |
|view      |medio_alto   |electronics       |event_type=view | price_segment=medio_alto | category=electronics|
|view      |medio_alto   |appliances        |event_type=view | price_segment=medio_alto | category=appli

## 8. Probabilidades de ocurrencia de combinaciones

La probabilidad de ocurrencia de cada partición se calcula como:

$$
P(\text{partición}) = \frac{\text{número de registros en la partición}}{\text{número total de registros}}
$$

Esta tabla documenta las combinaciones de particionamiento generadas por las variables seleccionadas.

In [11]:
total_registros = df_part.count()

partition_stats = (
    df_part
    .groupBy("event_type", "price_segment", "category_partition", "partition_rule")
    .agg(
        F.count("*").alias("n_registros"),
        F.countDistinct("user_id").alias("n_usuarios"),
        F.avg("price").alias("precio_promedio"),
        F.stddev("price").alias("desv_precio")
    )
    .withColumn("probabilidad_ocurrencia", F.col("n_registros") / F.lit(total_registros))
    .orderBy(F.desc("n_registros"))
)

partition_stats.show(50, truncate=False)

+----------+-------------+------------------+---------------------------------------------------------------------+-----------+----------+---------------+------------------+-----------------------+
|event_type|price_segment|category_partition|partition_rule                                                       |n_registros|n_usuarios|precio_promedio|desv_precio       |probabilidad_ocurrencia|
+----------+-------------+------------------+---------------------------------------------------------------------+-----------+----------+---------------+------------------+-----------------------+
|view      |alto         |electronics       |event_type=view | price_segment=alto | category=electronics          |63108      |55684     |767.228286     |403.0801575826387 |0.14837699437132337    |
|view      |bajo         |unknown           |event_type=view | price_segment=bajo | category=unknown              |57441      |50604     |34.034821      |17.808666428690685|0.13505297163090552    |
|view     

## 9. Selección de particiones objetivo

Para mostrar ejemplos detallados, se seleccionan las particiones con mayor frecuencia. En un proyecto real, también podrían seleccionarse particiones de interés estratégico, por ejemplo, compras de precio alto o categorías prioritarias.

In [12]:
NUM_PARTICIONES_EJEMPLO = 12

particiones_objetivo = (
    partition_stats
    .limit(NUM_PARTICIONES_EJEMPLO)
    .select("partition_rule")
)

particiones_objetivo_list = [r["partition_rule"] for r in particiones_objetivo.collect()]
particiones_objetivo_list

['event_type=view | price_segment=alto | category=electronics',
 'event_type=view | price_segment=bajo | category=unknown',
 'event_type=view | price_segment=medio_alto | category=electronics',
 'event_type=view | price_segment=medio_bajo | category=unknown',
 'event_type=view | price_segment=medio_bajo | category=electronics',
 'event_type=view | price_segment=medio_alto | category=unknown',
 'event_type=view | price_segment=alto | category=unknown',
 'event_type=view | price_segment=bajo | category=appliances',
 'event_type=view | price_segment=medio_alto | category=appliances',
 'event_type=view | price_segment=alto | category=computers',
 'event_type=view | price_segment=bajo | category=electronics',
 'event_type=view | price_segment=alto | category=appliances']

## 10. Extracción de submuestras por regla de particionamiento

A continuación se implementa un procedimiento para recuperar registros de cada partición. Esta sección funciona como evidencia de que el código puede filtrar la base de datos original y generar subconjuntos de acuerdo con las reglas construidas.

In [13]:
def extraer_particion(df_base, regla, n=10):
    """
    Extrae una submuestra de ejemplo para una regla de particionamiento.

    Parámetros:
    df_base: DataFrame de PySpark con columna partition_rule.
    regla: texto exacto de la regla de particionamiento.
    n: número de registros a recuperar.

    Retorna:
    DataFrame filtrado.
    """
    return (
        df_base
        .filter(F.col("partition_rule") == regla)
        .select(
            "event_time", "event_type", "product_id", "category_code",
            "category_partition", "price", "price_segment", "user_id",
            "user_session", "hour", "day_of_week", "partition_rule"
        )
        .limit(n)
    )


In [14]:
# Guardar muestras ejemplo de cada partición objetivo en una carpeta local.
# Se exporta con Pandas para evitar el error HADOOP_HOME/winutils en Windows.
import os

OUTPUT_DIR = "data/processed/particiones_ejemplo"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for i, regla in enumerate(particiones_objetivo_list, start=1):
    muestra = extraer_particion(df_part, regla, n=100)
    safe_name = f"particion_{i:02d}"

    muestra_pd = muestra.toPandas()
    muestra_pd.to_csv(
        f"{OUTPUT_DIR}/{safe_name}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(f"{safe_name}: {len(muestra_pd)} registros exportados")

print(f"\nSubmuestras guardadas en: {OUTPUT_DIR}")


particion_01: 100 registros exportados
particion_02: 100 registros exportados
particion_03: 100 registros exportados
particion_04: 100 registros exportados
particion_05: 100 registros exportados
particion_06: 100 registros exportados
particion_07: 100 registros exportados
particion_08: 100 registros exportados
particion_09: 100 registros exportados
particion_10: 100 registros exportados
particion_11: 100 registros exportados
particion_12: 100 registros exportados

Submuestras guardadas en: data/processed/particiones_ejemplo


## 11. Técnica de muestreo propuesta

### Técnica seleccionada: muestreo estratificado con sobrerrepresentación controlada de particiones pequeñas

Se propone utilizar **muestreo estratificado** donde cada partición funciona como un estrato. La asignación base se calcula de forma proporcional a la frecuencia de cada partición en la población observada. Sin embargo, también se aplica un mínimo de registros por partición (`MIN_POR_PARTICION`) para evitar que eventos poco frecuentes, como `cart` o `purchase`, desaparezcan de la muestra.

Esta técnica es adecuada porque:

- conserva parcialmente la estructura de la población observada;
- evita que solo las particiones grandes dominen la muestra;
- garantiza presencia de particiones pequeñas cuando existe suficiente información;
- reduce sesgos frente a una muestra por conveniencia;
- permite controlar el tamaño final de la muestra para manejar restricciones computacionales.

Es importante aclarar que, al usar un mínimo por partición, la muestra deja de ser estrictamente proporcional. Por ello, esta muestra es útil para análisis exploratorio y comparación entre particiones, pero las proporciones poblacionales deben interpretarse con cuidado.

Como el dataset es de Big Data y puede contener sesgos de selección propios de datos transaccionales, la muestra debe interpretarse como representativa de la **población observada en la plataforma**, no necesariamente de todos los clientes potenciales del mercado.

In [15]:
# Parámetros de muestreo
SAMPLE_FRACTION_GLOBAL = 0.01   # 1% de la población observada
MIN_POR_PARTICION = 30          # mínimo deseado por partición, si existe suficiente información
SEED = 42

# Tamaño objetivo total por restricción de recursos computacionales
n_total = max(1000, int(total_registros * SAMPLE_FRACTION_GLOBAL))
print(f"Tamaño objetivo aproximado de muestra: {n_total:,}")

Tamaño objetivo aproximado de muestra: 4,253


In [16]:
# Asignación proporcional por partición
allocation = (
    partition_stats
    .withColumn("n_muestra_proporcional", F.round(F.col("probabilidad_ocurrencia") * F.lit(n_total)).cast("int"))
    .withColumn(
        "n_muestra",
        F.when(F.col("n_registros") >= MIN_POR_PARTICION,
               F.greatest(F.col("n_muestra_proporcional"), F.lit(MIN_POR_PARTICION)))
         .otherwise(F.col("n_registros").cast("int"))
    )
    .withColumn("fraccion_muestreo", F.col("n_muestra") / F.col("n_registros"))
    .select("partition_rule", "n_registros", "probabilidad_ocurrencia", "n_muestra", "fraccion_muestreo")
)

allocation.orderBy(F.desc("n_registros")).show(50, truncate=False)

+---------------------------------------------------------------------+-----------+-----------------------+---------+--------------------+
|partition_rule                                                       |n_registros|probabilidad_ocurrencia|n_muestra|fraccion_muestreo   |
+---------------------------------------------------------------------+-----------+-----------------------+---------+--------------------+
|event_type=view | price_segment=alto | category=electronics          |63108      |0.14837699437132337    |631      |0.009998732331875515|
|event_type=view | price_segment=bajo | category=unknown              |57441      |0.13505297163090552    |574      |0.009992862241256246|
|event_type=view | price_segment=medio_alto | category=electronics    |48080      |0.11304376448902244    |481      |0.010004159733777038|
|event_type=view | price_segment=medio_bajo | category=unknown        |31020      |0.07293297783796747    |310      |0.009993552546744036|
|event_type=view | price_se

### Opción alternativa: muestreo estrictamente proporcional

Si se desea una muestra que conserve mejor las proporciones originales de la población observada, puede usarse una asignación estrictamente proporcional sin mínimo por partición. Esta alternativa es más representativa para estimar proporciones, pero puede dejar con pocos registros a eventos raros.

In [17]:
# Opción alternativa: asignación estrictamente proporcional sin mínimo por partición
# No es obligatorio ejecutarla si se mantiene el diseño con mínimo por partición.

allocation_proporcional = (
    partition_stats
    .withColumn("n_muestra", F.round(F.col("probabilidad_ocurrencia") * F.lit(n_total)).cast("int"))
    .withColumn("n_muestra", F.when(F.col("n_muestra") < 1, F.lit(1)).otherwise(F.col("n_muestra")))
    .withColumn("fraccion_muestreo", F.col("n_muestra") / F.col("n_registros"))
    .select("partition_rule", "n_registros", "probabilidad_ocurrencia", "n_muestra", "fraccion_muestreo")
)

allocation_proporcional.orderBy(F.desc("n_registros")).show(20, truncate=False)


+-----------------------------------------------------------------+-----------+-----------------------+---------+--------------------+
|partition_rule                                                   |n_registros|probabilidad_ocurrencia|n_muestra|fraccion_muestreo   |
+-----------------------------------------------------------------+-----------+-----------------------+---------+--------------------+
|event_type=view | price_segment=alto | category=electronics      |63108      |0.14837699437132337    |631      |0.009998732331875515|
|event_type=view | price_segment=bajo | category=unknown          |57441      |0.13505297163090552    |574      |0.009992862241256246|
|event_type=view | price_segment=medio_alto | category=electronics|48080      |0.11304376448902244    |481      |0.010004159733777038|
|event_type=view | price_segment=medio_bajo | category=unknown    |31020      |0.07293297783796747    |310      |0.009993552546744036|
|event_type=view | price_segment=medio_bajo | category=

## 12. Implementación del muestreo estratificado con mínimo por partición

PySpark permite aplicar muestreo por estratos mediante `sampleBy`. Para ello se construye un diccionario con las fracciones de muestreo por partición. La columna `partition_rule` debe existir previamente en `df_part`, ya que es la variable que identifica cada estrato.

In [18]:
fractions = {
    row["partition_rule"]: min(float(row["fraccion_muestreo"]), 1.0)
    for row in allocation.collect()
}

# df_part debe contener la columna partition_rule. Esta celda usa el df_part creado en la sección 7.
sample_stratified = df_part.sampleBy("partition_rule", fractions=fractions, seed=SEED)

print(f"Tamaño real de la muestra estratificada: {sample_stratified.count():,}")
sample_stratified.show(5, truncate=False)


Tamaño real de la muestra estratificada: 5,844
+-------------------+----------+----------+-------------------+----------------------------+--------+------+---------+------------------------------------+----------+----+-----------+----------------+-------------+------------------+------------------------------------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code               |brand   |price |user_id  |user_session                        |event_date|hour|day_of_week|category_level_1|price_segment|category_partition|partition_rule                                                    |
+-------------------+----------+----------+-------------------+----------------------------+--------+------+---------+------------------------------------+----------+----+-----------+----------------+-------------+------------------+------------------------------------------------------------------+
|2019-09-30 19:57:59|view      |42400035  |2095518

In [19]:
sample_summary = (
    sample_stratified
    .groupBy("partition_rule")
    .agg(F.count("*").alias("n_muestra_real"))
    .join(allocation, on="partition_rule", how="right")
    .fillna({"n_muestra_real": 0})
    .withColumn("diferencia_vs_objetivo", F.col("n_muestra_real") - F.col("n_muestra"))
    .orderBy(F.desc("n_registros"))
)

sample_summary.show(50, truncate=False)

+---------------------------------------------------------------------+--------------+-----------+-----------------------+---------+--------------------+----------------------+
|partition_rule                                                       |n_muestra_real|n_registros|probabilidad_ocurrencia|n_muestra|fraccion_muestreo   |diferencia_vs_objetivo|
+---------------------------------------------------------------------+--------------+-----------+-----------------------+---------+--------------------+----------------------+
|event_type=view | price_segment=alto | category=electronics          |619           |63108      |0.14837699437132337    |631      |0.009998732331875515|-12                   |
|event_type=view | price_segment=bajo | category=unknown              |568           |57441      |0.13505297163090552    |574      |0.009992862241256246|-6                    |
|event_type=view | price_segment=medio_alto | category=electronics    |524           |48080      |0.113043764489022

## 13. Validación de representatividad de la muestra

Para verificar el comportamiento de la muestra, se comparan proporciones de variables clave entre la base completa y la muestra. Esta validación permite identificar si la muestra conserva la estructura general o si alguna variable quedó sobre/subrepresentada.

Debido al uso de un mínimo por partición, es esperable que clases minoritarias como `cart` o `purchase` aparezcan con mayor proporción en la muestra que en la población observada. Esto no necesariamente es un error: es una consecuencia del diseño de muestreo elegido para asegurar que los eventos menos frecuentes estén presentes en el análisis.

In [20]:
def comparar_distribucion(df_poblacion, df_muestra, columna):
    pop = (
        df_poblacion.groupBy(columna).count()
        .withColumnRenamed("count", "n_poblacion")
    )
    samp = (
        df_muestra.groupBy(columna).count()
        .withColumnRenamed("count", "n_muestra")
    )
    total_pop = df_poblacion.count()
    total_samp = df_muestra.count()

    return (
        pop.join(samp, on=columna, how="outer")
        .fillna(0)
        .withColumn("prop_poblacion", F.col("n_poblacion") / F.lit(total_pop))
        .withColumn("prop_muestra", F.col("n_muestra") / F.lit(total_samp))
        .withColumn("diferencia_abs", F.abs(F.col("prop_poblacion") - F.col("prop_muestra")))
        .orderBy(F.desc("prop_poblacion"))
    )

for columna in ["event_type", "price_segment", "category_partition"]:
    print(f"\nComparación para {columna}")
    comparar_distribucion(df_part, sample_stratified, columna).show(50, truncate=False)


Comparación para event_type
+----------+-----------+---------+--------------------+------------------+-------------------+
|event_type|n_poblacion|n_muestra|prop_poblacion      |prop_muestra      |diferencia_abs     |
+----------+-----------+---------+--------------------+------------------+-------------------+
|view      |408827     |4319     |0.9612176186512806  |0.7390485968514716|0.22216902179980902|
|cart      |9110       |703      |0.021419066025270266|0.1202943189596167|0.09887525293434643|
|purchase  |7385       |822      |0.017363315323449056|0.1406570841889117|0.12329376886546264|
+----------+-----------+---------+--------------------+------------------+-------------------+


Comparación para price_segment
+-------------+-----------+---------+-------------------+-------------------+--------------------+
|price_segment|n_poblacion|n_muestra|prop_poblacion     |prop_muestra       |diferencia_abs      |
+-------------+-----------+---------+-------------------+------------------

### Interpretación de la validación

Si la distribución de `event_type` en la muestra no coincide exactamente con la población, esto se debe al mínimo de registros por partición. En este dataset, `view` suele dominar la población, mientras que `cart` y `purchase` aparecen con menor frecuencia. Al forzar un mínimo por partición, la muestra aumenta la presencia relativa de esos eventos minoritarios.

Por lo tanto, la muestra estratificada con mínimo por partición es adecuada para **comparar comportamientos entre tipos de evento y particiones**, pero no debe usarse directamente para estimar proporciones poblacionales exactas sin considerar ponderaciones.

Una alternativa estrictamente proporcional sería eliminar el mínimo por partición y usar solamente la fracción calculada a partir de la probabilidad de ocurrencia. Sin embargo, esa opción podría dejar con muy pocos registros a las particiones pequeñas.

## 14. Muestreo alternativo: muestra balanceada para comparación entre particiones

Además del muestreo estratificado, se crea una muestra balanceada para análisis donde se desea comparar particiones con tamaños similares. Esta muestra toma hasta `N_BALANCEADO_POR_PARTICION` registros por cada regla de particionamiento.

La muestra balanceada **no conserva la distribución original** de la población. Su objetivo es comparativo: permite observar particiones frecuentes y poco frecuentes con pesos similares. Por ello, no debe usarse para estimar proporciones poblacionales, pero sí puede ser útil para análisis exploratorio, comparación entre grupos o entrenamiento preliminar de modelos cuando existe fuerte desbalance.

In [21]:
N_BALANCEADO_POR_PARTICION = 100

w = Window.partitionBy("partition_rule").orderBy(F.rand(SEED))

sample_balanced = (
    df_part
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") <= N_BALANCEADO_POR_PARTICION)
    .drop("rn")
)

print(f"Tamaño de muestra balanceada: {sample_balanced.count():,}")
sample_balanced.groupBy("partition_rule").count().orderBy(F.desc("count")).show(50, truncate=False)

Tamaño de muestra balanceada: 7,047
+---------------------------------------------------------------------+-----+
|partition_rule                                                       |count|
+---------------------------------------------------------------------+-----+
|event_type=view | price_segment=alto | category=electronics          |100  |
|event_type=view | price_segment=medio_bajo | category=otras          |100  |
|event_type=view | price_segment=alto | category=furniture            |100  |
|event_type=cart | price_segment=alto | category=computers            |100  |
|event_type=view | price_segment=alto | category=otras                |100  |
|event_type=cart | price_segment=alto | category=unknown              |100  |
|event_type=view | price_segment=alto | category=unknown              |100  |
|event_type=cart | price_segment=bajo | category=electronics          |100  |
|event_type=view | price_segment=bajo | category=apparel              |100  |
|event_type=cart | price_seg

## 15. Guardado de resultados

Se guardan tres salidas:

1. Tabla de caracterización de variables.
2. Tabla de reglas de particionamiento y probabilidades.
3. Muestra estratificada proporcional.

In [22]:
# Guardado de resultados en CSV usando Pandas para evitar problemas de Hadoop/winutils en Windows.
import os

PROCESSED_DIR = "data/processed/evidencia1"
os.makedirs(PROCESSED_DIR, exist_ok=True)

caracterizacion_df.toPandas().to_csv(
    f"{PROCESSED_DIR}/caracterizacion_variables.csv",
    index=False,
    encoding="utf-8-sig"
)

partition_stats.limit(5000).toPandas().to_csv(
    f"{PROCESSED_DIR}/reglas_particionamiento.csv",
    index=False,
    encoding="utf-8-sig"
)

sample_stratified.limit(10000).toPandas().to_csv(
    f"{PROCESSED_DIR}/muestra_estratificada.csv",
    index=False,
    encoding="utf-8-sig"
)

sample_balanced.limit(10000).toPandas().to_csv(
    f"{PROCESSED_DIR}/muestra_balanceada.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Resultados guardados en: {PROCESSED_DIR}")


Resultados guardados en: data/processed/evidencia1


## 16. Conclusiones

En este notebook se definió una estrategia de muestreo para una base de datos de comportamiento de comercio electrónico. Primero se caracterizó la población observada mediante variables de comportamiento, precio, categoría y temporalidad. Después se construyeron reglas de particionamiento combinando `event_type`, `price_segment` y `category_partition`, lo que permitió calcular probabilidades de ocurrencia para cada subconjunto.

La técnica principal propuesta fue el muestreo estratificado con sobrerrepresentación controlada de particiones pequeñas. Esta decisión permite que eventos minoritarios, como `cart` y `purchase`, tengan suficiente presencia en la muestra para ser analizados. Sin embargo, también implica que la muestra no conserva exactamente las proporciones originales de la población, por lo que los resultados deben interpretarse como una muestra útil para comparación y análisis exploratorio.

También se incluyó una muestra balanceada por partición. Esta alternativa es adecuada cuando se busca comparar grupos con tamaños similares, pero no debe interpretarse como representativa de la distribución original. En ambos casos, la aleatorización dentro de cada partición ayuda a reducir sesgos frente a una extracción manual o por conveniencia.

Finalmente, se aclara que la representatividad se limita a la población observada en el dataset, ya que los datos transaccionales de Big Data pueden presentar sesgos de selección. Por ello, las conclusiones deben interpretarse dentro del contexto de usuarios y eventos registrados por la plataforma.

## 18. Recomendaciones finales para interpretar los resultados

- Si el objetivo es estimar proporciones reales de la población observada, conviene usar una muestra estrictamente proporcional o aplicar ponderaciones.
- Si el objetivo es comparar particiones, la muestra con mínimo por partición y la muestra balanceada son adecuadas.
- Las categorías `unknown` en `category_partition` son esperadas porque `category_code` contiene valores nulos en el dataset original.
- La dominancia de `view` sobre `cart` y `purchase` es normal en datos de e-commerce, ya que muchos usuarios visualizan productos sin agregarlos al carrito o comprarlos.
- La exportación con Pandas se usa únicamente por compatibilidad con Windows local y para evitar problemas de Hadoop/winutils al escribir archivos.

## 17. Bibliografía

- Ahmed, S. K. (2024). *How to choose a sampling technique and determine sample size for research: A simplified guide for researchers*. Oral Oncology Reports, 12, 100662.
- Kim, J. K., & Wang, Z. (2019). *Sampling Techniques for Big Data Analysis*. International Statistical Review, 87(S1), S177–S191.
- Lakens, D. (2022). *Sample Size Justification*. Collabra: Psychology, 8(1).
- Kaggle. (s. f.). *eCommerce behavior data from multi category store*. Dataset de Kaggle.
- Apache Spark. (s. f.). *PySpark Documentation*. Apache Spark.